# Day 22: Text‑to‑Image with ControlNet (Sketch Conditioning)

In [ ]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from PIL import Image
import numpy as np
import cv2
import requests
from io import BytesIO
import matplotlib.pyplot as plt

## 1. Load ControlNet (sketch version)
We'll use a ControlNet trained on sketch‑like edges (Canny or lineart).

In [ ]:
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", controlnet=controlnet, torch_dtype=torch.float16
).to("cuda" if torch.cuda.is_available() else "cpu")
pipe.safety_checker = None  # disable for speed
print("Pipeline loaded")

## 2. Create or load a sketch image
We'll generate a simple sketch (a cat outline) programmatically, then detect edges to use as control.

In [ ]:
# Create a blank canvas
sketch = np.zeros((512, 512, 3), dtype=np.uint8)
# Draw a rough cat shape (using OpenCV)
cv2.ellipse(sketch, (256, 300), (80, 100), 0, 0, 360, (255,255,255), 3)  # body
cv2.circle(sketch, (210, 230), 30, (255,255,255), 3)  # left ear region
cv2.circle(sketch, (302, 230), 30, (255,255,255), 3)  # right ear
cv2.ellipse(sketch, (256, 280), (40, 30), 0, 0, 360, (255,255,255), 2)  # face

plt.imshow(sketch)
plt.title("Input Sketch")
plt.show()

# Convert to PIL and run edge detection (Canny)
sketch_pil = Image.fromarray(sketch)
sketch_gray = np.array(sketch_pil.convert("L"))
edges = cv2.Canny(sketch_gray, 50, 150)
control_image = Image.fromarray(edges)
plt.imshow(control_image, cmap='gray')
plt.title("Control Image (Canny edges)")
plt.show()

## 3. Generate image conditioned on sketch + prompt

In [ ]:
prompt = "a cute cat, detailed fur, fluffy, orange tabby, high quality"
negative_prompt = "ugly, deformed, blurry"

with torch.no_grad():
    image = pipe(
        prompt,
        negative_prompt=negative_prompt,
        image=control_image,
        num_inference_steps=20,
        guidance_scale=7.0,
        controlnet_conditioning_scale=0.8
    ).images[0]

image

## 4. (Optional) IP‑Adapter for image prompt
IP‑Adapter allows you to use a reference image as a style/subject prompt. This requires a separate pipeline. We'll show a minimal example.

In [ ]:
# Uncomment if you have IP-Adapter installed
# from diffusers import StableDiffusionPipeline
# from ip_adapter import IPAdapter
# 
# sd_pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")
# ip_model = IPAdapter(sd_pipe, "ip-adapter_sd15.bin", device="cuda")
# reference_image = Image.open("style.jpg")
# images = ip_model.generate(prompt="cat", image=reference_image, scale=0.6)
print("IP‑Adapter example requires additional setup – see exercise 5.")